# Genre Data Collection

This was done separately from the others because of the difficulty I have in retrieving the songs genre. Unlike the other features, spotify attributes genre to artist and the song. I have to get the artist id first before getting the genre. And I could only do that using sp.artist() and request. This method using request uses an access token goten from Spotify Developer Console

In [1]:
import requests
import pandas as pd
import numpy as np
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import time

In [2]:
gym = pd.read_csv('/Users/mac/Downloads/gym_data.csv')
bedroom = pd.read_csv('/Users/mac/Downloads/bedroom_data.csv')
library = pd.read_csv('/Users/mac/Downloads/library_data.csv')
office = pd.read_csv('/Users/mac/Downloads/office_data.csv')
trainBus = pd.read_csv('/Users/mac/Downloads/trainbus_data.csv')
restaurant = pd.read_csv('/Users/mac/Downloads/restaurant_data.csv')
bar = pd.read_csv('/Users/mac/Downloads/bar_data.csv')
mall = pd.read_csv('/Users/mac/Downloads/mall_data.csv')

In [ ]:
def get_artist_id_batch(track_ids, access_token, batch_size=70, delay=120):
    # Initialize an empty list to store the artist IDs
    artist_ids = []
    
    # Loop through track IDs in batches of specified 'batch_size' (default is 70)
    for i in range(0, len(track_ids), batch_size):
        # Create a batch by slicing the list of track IDs from 'i' to 'i + batch_size'
        batch = track_ids[i:i + batch_size]
        
        # Loop through each track ID in the current batch
        for track_id in batch:
            # Construct the Spotify API URL to fetch track information
            url = f"https://api.spotify.com/v1/tracks/{track_id}"
            
            # Set the authorization header with the provided access token
            headers = {
                "Authorization": f"Bearer {access_token}"
            }
            
            # Make a GET request to the Spotify API for the track information
            response = requests.get(url, headers=headers)
            
            # If the request is successful (status code 200), extract the artist ID
            if response.status_code == 200:
                track_data = response.json()
                artist_id = track_data['artists'][0]['id']  # Get the ID of the first artist
                artist_ids.append(artist_id)
            else:
                # If the request fails, log a message and append None to indicate missing data
                print(f"Failed to retrieve data for track ID {track_id}")
                artist_ids.append(None)
        
        # Pause the process for a specified delay time (default is 120 seconds) to avoid hitting rate limits
        time.sleep(delay)
    
    # Return the list of collected artist IDs
    return artist_ids

In [ ]:
access_token="XXXXX"

In [ ]:
start_index = 560
end_index = start_index + len(artist_ids)  # Calculate the end index based on the length of artist_ids

# Check if the range is correct
if len(artist_ids) == len(bedroom.loc[start_index:]):
    # Assign the artist_ids list to the specified range in the DataFrame
    bedroom.loc[start_index:, 'artist_id'] = artist_ids
else:
    print("Length of artist_ids does not match the number of rows from index 560 to the end.")


In [ ]:
start_index = 760
end_index = start_index + len(new_artist_ids)  # Calculate the end index based on the length of artist_ids

# Check if the range is correct
if len(new_artist_ids) == len(bedroom.loc[start_index:]):
    # Assign the artist_ids list to the specified range in the DataFrame
    bedroom.loc[start_index:, 'artist_id'] = new_artist_ids
else:
    print("Length of artist_ids does not match the number of rows from index 760 to the end.")

In [ ]:
# Get Artist ID for all the songs in the locations

In [ ]:
gym['artist_id'] = get_artist_id_batch(gym['track_id'].tolist(), access_token, batch_size=10, delay=2)

bedroom['artist_id'] = get_artist_id_batch(bedroom['track_id'].tolist(), access_token, batch_size=70, delay=120)

In [ ]:
library['artist_id'] = get_artist_id_batch(library['track_id'].tolist(), access_token, batch_size=100, delay=120)

In [ ]:
office['artist_id'] = get_artist_id_batch(office['track_id'].tolist(), access_token, batch_size=100, delay=120)

In [ ]:
trainBus['artist_id'] = get_artist_id_batch(trainBus['track_id'].tolist(), access_token, batch_size=100, delay=120)

In [ ]:
mall['artist_id'] = get_artist_id_batch(mall['track_id'].tolist(), access_token, batch_size=100, delay=120)

In [ ]:
restaurant['artist_id'] = get_artist_id_batch(restaurant['track_id'].tolist(), access_token, batch_size=100, delay=120)

In [ ]:
bar['artist_id'] = get_artist_id_batch(bar['track_id'].tolist(), access_token, batch_size=100, delay=120)

# After collecting the artist id, created a function get_artist_genre_batch to use the artist id and fetch the genre

In [4]:
def get_artist_genre_batch(artist_ids, access_token, batch_size=10, delay=2):
    """
    Retrieves genres for a list of artists in batches from the Spotify API.
    
    Args:
        artist_ids (list): A list of artist IDs for which genres are to be retrieved.
        access_token (str): A valid Spotify API access token used for authentication.
        batch_size (int, optional): The number of artist IDs to process per batch. Default is 10.
        delay (int, optional): The delay time in seconds between each batch to avoid rate limiting. Default is 2 seconds.
    
    Returns:
        list: A list of genres for each artist, where each element is a list of genres or None if the data could not be retrieved.
    """
    # Initialize an empty list to store the genres for each artist
    genres_list = []
    
    # Loop through artist IDs in batches of specified 'batch_size' (default is 10)
    for i in range(0, len(artist_ids), batch_size):
        # Create a batch by slicing the list of artist IDs from 'i' to 'i + batch_size'
        batch = artist_ids[i:i + batch_size]
        
        # Loop through each artist ID in the current batch
        for artist_id in batch:
            # Construct the Spotify API URL to fetch artist information
            url = f"https://api.spotify.com/v1/artists/{artist_id}"
            
            # Set the authorization header with the provided access token
            headers = {
                "Authorization": f"Bearer {access_token}"
            }
            
            # Make a GET request to the Spotify API for the artist information
            response = requests.get(url, headers=headers)
            
            # If the request is successful (status code 200), extract the artist's genre information
            if response.status_code == 200:
                artist_data = response.json()
                genres = artist_data.get('genres', [])  # Get the genres list (empty list if no genres)
                genres_list.append(genres)
            else:
                # If the request fails, log a message and append None to indicate missing data
                print(f"Failed to retrieve data for artist ID {artist_id}")
                genres_list.append(None)
        
        # Pause between processing each batch to avoid rate limiting (default delay is 2 seconds)
        time.sleep(delay)
    
    # Return the list of genres for each artist ID
    return genres_list


In [ ]:
access_token = "XXXXXX"

In [ ]:
bar['genre'] = get_artist_genre_batch(bar['artist_id'].tolist(), access_token, batch_size=100, delay=120)

In [ ]:
mall['genre'] = get_artist_genre_batch(mall['artist_id'].tolist(), access_token, batch_size=100, delay=120)

In [ ]:
restaurant['genre'] = get_artist_genre_batch(restaurant['artist_id'].tolist(), access_token, batch_size=100, delay=120)

In [ ]:
trainBus['genre'] = get_artist_genre_batch(trainBus['artist_id'].tolist(), access_token, batch_size=100, delay=120)

In [ ]:
office['genre'] = get_artist_genre_batch(office['artist_id'].tolist(), access_token, batch_size=100, delay=120)

In [ ]:
access_token = "XXXXX"

In [ ]:
multiple_artist_ids = office['artist_id'].loc[500:].tolist()  # Example range
genre_list = get_artist_genre_batch(multiple_artist_ids, access_token, batch_size=100, delay=120)

In [ ]:
start_index = 500
# Convert list of genres to a string with genres separated by commas
genre_list = [", ".join(genres) if genres else None for genres in genre_list]

# Now you can safely assign it to the DataFrame
if len(genre_list) == len(office.loc[start_index:]):
    office.loc[start_index:, 'genre'] = genre_list
else:
    print("Length of genre_list does not match the number of rows from index 500 to the end.")

In [ ]:
access_token = "XXXXX"

In [10]:
library['genre'] = get_artist_genre_batch(library['artist_id'].tolist(), access_token, batch_size=100, delay=120)

In [ ]:
access_token = 'XXXXX'

In [13]:
gym['genre'] = get_artist_genre_batch(gym['artist_id'].tolist(), access_token, batch_size=100, delay=120)

In [ ]:
access_token = "XXXX"

In [16]:
bedroom['genre'] = get_artist_genre_batch(bedroom['artist_id'].tolist(), access_token, batch_size=100, delay=120)

In [ ]:
trainBus.to_csv('trainbus.csv', index=False)
office.to_csv('office.csv', index=False)
mall.to_csv('mall.csv', index=False)
restaurant.to_csv('restaurant.csv', index=False)
bar.to_csv('bar.csv', index=False)
bedroom.to_csv('bedroom_gen.csv', index=False)
gym.to_csv('gym.csv', index=False)
library.to_csv('library.csv', index=False)